# Pipeline for genData from scraper to train model

In [ ]:
import spacy
import json
from spacy.training.example import Example
import matplotlib.pyplot as plt
from spacy.matcher import PhraseMatcher


In [ ]:
#load our model
nlp = spacy.load("../models/trained_ner_model_best")
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

In [ ]:
# Load skills from a linkedin skills file
skill_file = "../data/linkedin_skills.txt"

with open(skill_file, "r", encoding="utf-8") as f:
    skill_list = [line.strip() for line in f.readlines() if line.strip()]

In [ ]:
# Load your training data.
with open("../data/gendata.json", "r") as f:
    training_data = json.load(f)

In [ ]:
# Function to clean skills (Fix HTML entities, handle hyphens, preserve single letters)
def clean_skill(skill):
    skill = html.unescape(skill)  # Convert HTML entities (&amp; -> &)
    skill = skill.replace("\t", " ").strip()  # Remove tabs and extra spaces
    skill = re.sub(r"\s+", " ", skill)  # Normalize multiple spaces

    # Normalize ampersands to "and"
    skill = skill.replace("&", "and")  

    # Convert hyphens to spaces for better tokenization
    skill = skill.replace("-", " ")  

    # Preserve single-letter words (e.g., "v" in "hyper v") by adding "_"
    skill = re.sub(r"\b([a-zA-Z])\b", r"\1_", skill)  

    # Normalize apostrophes to avoid tokenization errors
    skill = skill.replace("’", "'")  # Normalize different apostrophe characters

    return skill.lower().strip()  # Convert to lowercase for better matching

In [ ]:
# Apply cleaning function
skill_list = [clean_skill(skill) for skill in skill_list]

# ✅ Use PhraseMatcher to add skills for proper tokenization
skill_patterns = [nlp.make_doc(skill) for skill in skill_list]
matcher.add("SKILL", skill_patterns)

In [ ]:
def generate_training_data(sentence_templates, num_sentences=5000):
    """Generates labeled training data for spaCy's NER model"""
    training_data = []
    used_sentences = set()

    for sentence in sentence_templates:
        
        # Ensure uniqueness to prevent duplicate patterns
        if sentence in used_sentences:
            continue
        used_sentences.add(sentence)

        # Tokenize sentence using spaCy's tokenizer
        doc = nlp(sentence)


        # Ensure skills like "hyper v" are matched correctly
        matches = matcher(doc)
        matched_entities = []
        for match_id, start, end in matches:
            span = doc[start:end]
            
            matched_entities.append((span.start_char, span.end_char, "SKILL"))

        if not matched_entities:
            print(f"Skipping misaligned skill: '{skill}' in sentence: '{sentence}'")
            continue  # Skip misaligned skill

        training_data.append((sentence, {"entities": matched_entities}))

    return training_data

In [ ]:
# Generate training data
training_data = generate_training_data(skill_list, training_data, num_sentences=20000)